# 2 · Build a dataset

Start with ten labels from notebook 0. Find a Sentinel-2 image for each label's place and day, then save each image/label pair as one STAC Item:

```text
data/my_dataset/
├── train/  <tile>/B02.tif  B03.tif  B04.tif  B08.tif  label.tif
├── val/    ...
├── test/   ...
└── catalog.parquet          one row per pair, with its split
```

We will also save six forest scenes for prediction in notebook 3. This notebook needs internet access.

**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.

**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync`, internet access, and notebook 0's download.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml
from dask.diagnostics.progress import ProgressBar

import geosave as gs

# Show progress when reading pixels.
pbar = ProgressBar(minimum=1)
pbar.register()

root = Path("data/dynamic_world")
classes = yaml.safe_load((root / "classes.yaml").read_text())

## Read a label's place and date

Its `region` describes geometry and time for searching. Its `geobox` defines the exact pixel grid for loading.

In [ ]:
label_path = sorted((root / "train/labels").glob("*.tif"))[0]
label = gs.open_raster(label_path)
label = label.gs.write_flags(classes["class_map"], classes["color_map"])

region = label.gs.region
grid = label.gs.geobox
place = region.format("{lat:.2f}, {lon:.2f} · {start:%Y-%m-%d} · {width:g} m across")
f"{place} at {abs(grid.resolution.x):g} m"

## Find the image

A [STAC](https://stacspec.org) catalog lists scenes by place and date. Ask for the least cloudy Sentinel-2 scene and keep one. Search the label's own day so the image matches the label; this reads metadata, not pixels.

In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]

client = gs.StacClient.open("planetary_computer")
query = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=1)
query = query.sort_by("properties.eo:cloud_cover")

day = region.replace(timespan=region.format("{start:%Y-%m-%d}"))
items = client.search_items(query.set_region(day))
for item in items:
    print(item.id, item.properties["eo:cloud_cover"])

## Load it on the label's grid

`gs.stac.load_dataset` loads the selected Items onto a grid you choose; `chunks={}` keeps the read lazy. Planetary Computer does not state how its stored numbers become reflectance, so record it with `.gs.write_packing`. The dates in this lesson are 2019, before the January 2022 Sentinel-2 offset change: reflectance = DN × 1e-4. Verify scale and offset before reusing this for another product or year; a wrong offset changes every model input.

In [ ]:
PACKING = gs.attrs.CFPacking(scale_factor=1e-4, add_offset=0, fill_value=0)


def load_image(items, grid):
    image = gs.stac.load_dataset(
        items,
        geobox=grid,
        bands=BANDS,
        dtype="uint16",
        nodata=0,
        groupby="solar_day",
        chunks={},
        patch_url=client.patch_url,
    )
    return image.gs.write_packing(PACKING)


image = load_image(items, grid)
image.isel(time=0).gs.plot(xlabel="") + label.gs.plot(xlabel="")

## Save the pair as one Item

Image and label share one grid, so they fit in one Dataset. `to_catalog` writes one Cloud-Optimized GeoTIFF per variable, `<id>/<variable>.tif`, and returns a catalog: a GeoDataFrame with one STAC Item per date. Name the Item after the label file.

A catalog stores file paths and metadata, not pixels. The Item keeps the Sentinel-2 Collection it was loaded from; the label is one more asset.

In [ ]:
folder = Path("data/my_dataset")

pair = image.assign(label=label["label"].expand_dims(time=image.time))
pair_catalog = pair.gs.to_catalog(
    folder / "train", id_format=label_path.stem, overwrite=True
)
pair_catalog[["id", "collection", "datetime"]]

## Build all three splits

`save_pair` repeats the steps above for one label file and returns its one-row catalog.

In [ ]:
def save_pair(label_path, folder):
    label = gs.open_raster(label_path)
    region = label.gs.region
    day = region.replace(timespan=region.format("{start:%Y-%m-%d}"))

    items = client.search_items(query.set_region(day))
    image = load_image(items, label.gs.geobox)

    pair = image.assign(label=label["label"].expand_dims(time=image.time))
    return pair.gs.to_catalog(folder, id_format=label_path.stem, overwrite=True)

`build_split` saves every pair of one split, joins their rows with `pd.concat`, and records the split as an ordinary column.

In [ ]:
from tqdm.auto import tqdm


def build_split(label_paths, split):
    catalogs = []
    for path in tqdm(label_paths, desc=split):
        catalogs.append(save_pair(path, folder / split))

    catalog = pd.concat(catalogs, ignore_index=True)
    catalog["split"] = split
    return catalog

Use six training labels, two validation labels and two test labels for this short example.

In [ ]:
pbar.unregister()

train = build_split(sorted((root / "train/labels").glob("*.tif"))[:6], "train")
val = build_split(sorted((root / "val/labels").glob("*.tif"))[:2], "val")
test = build_split(sorted((root / "test/labels").glob("*.tif"))[:2], "test")

catalog = pd.concat([train, val, test], ignore_index=True)
catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
catalog[["id", "split", "datetime"]]

### Optional: open the catalog viewer

Run this separately when you want a local viewer. Select rows, then use **Open selected items** to view their rasters. Stop it after browsing.

```python
explorer = catalog.gs.explore()
server = explorer.serve(threaded=True, open=False)
# When finished:
explorer.stop()
```

## Reopen a pair

The saved catalog is an ordinary GeoDataFrame. Select rows with pandas, then `.gs.to_dataset()` loads them: image bands and label on the pair's own grid.

In [ ]:
catalog = gs.open_vector(folder / "catalog.parquet")
rows = catalog[catalog["split"] == "val"]

rows.iloc[[0]].gs.to_dataset()

## Start from a shape

Open the forest outline in `khdtk.geojson` and give it a 10 m grid. Keep the least cloudy scene of each month, then load all six onto that grid in one call.

In [ ]:
from odc.geo.geobox import GeoBox

boundary = gs.open_vector("khdtk.geojson")
forest = gs.GeoRegion.from_geometry(boundary)
# KHDTK is in UTM zone 49S.
forest_grid = GeoBox.from_geopolygon(forest.extent, crs="EPSG:32749", resolution=10)

months = ["2019-05", "2019-06", "2019-07", "2019-08", "2019-09", "2019-10"]
scenes = []
for month in months:
    scenes.extend(client.search_items(query.set_region(forest.replace(timespan=month))))

season = load_image(scenes, forest_grid)
season

Save and catalog the six scenes for notebook 3. Each Item is named by its acquisition date; these are individual scenes, not monthly composites.

In [ ]:
khdtk = Path("data/khdtk")

season_catalog = season.gs.to_catalog(khdtk, id_format="{start:%Y%m%d}", overwrite=True)
season_catalog.gs.to_geoparquet(khdtk / "catalog.parquet", overwrite=True)
season_catalog[["id", "collection", "datetime"]]

## What happened

We saved image/label pairs in fixed splits and six forest scenes, each with a catalog. Notebook 3 loads both catalogs back with `.gs.to_dataset()` to train and to map.

## More things you can do with this dataset (optional)

The dataset and forest observations are saved. Continue to [notebook 3](03_inputs.ipynb), or try one of these extensions.

### Keep a shortlist when a label's scene is cloudy
Look at several candidates before downloading pixels. Keep the label's acquisition day fixed here; changing dates may change what the label describes.

In [ ]:
candidates = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=5)
candidates = candidates.sort_by("properties.eo:cloud_cover")

client.search(candidates.set_region(day))

### Expand the forest monitoring period
Add another month to `months`, reuse `forest_grid`, and inspect image quality before predicting it. Keep a record of missing observations rather than silently replacing them with another season.

### Work with a boundary your team recognizes
Replace `khdtk.geojson` with your project boundary and choose an appropriate projected grid. Keep the downloaded imagery and catalog together so another person can reopen the same observations.

### Compare greenness with the predicted land cover
When reviewing a predicted change in tree cover, compare it with NDVI from the same observation. This adds a review layer without changing the four-band model inputs.

In [ ]:
ndvi = gs.features.ndvi(season, nir="B08", red="B04")
season.assign(ndvi=ndvi).gs.explore("ndvi", basemap="Imagery", cmap="RdYlGn") * boundary.gs.explore()

**Next:** [train on these pairs and map the forest](03_inputs.ipynb). The forest scenes have no reference labels, so they are for application and visual review, not accuracy measurement.